In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 8fb926b5-4034-4f27-b12c-299bf43bac80, 3, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import current_timestamp, input_file_name

sources = {
    "bronze_products": "Files/landing/products/products.csv",
    "bronze_carriers": "Files/landing/carriers/carriers.csv",
    "bronze_warehouses": "Files/landing/warehouses/warehouses.csv",
    "bronze_shipments": "Files/landing/shipments/shipments.csv",
    "bronze_shipment_items": "Files/landing/shipment_items/shipment_items.csv",
    "bronze_sensor_readings": "Files/landing/sensor_readings/sensor_readings.csv"
}

for table_name, file_path in sources.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
        .withColumn("_ingested_at_utc", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Created {table_name}: {df.count()} rows")

StatementMeta(, 8fb926b5-4034-4f27-b12c-299bf43bac80, 5, Finished, Available, Finished, False)

Created bronze_products: 8 rows
Created bronze_carriers: 4 rows
Created bronze_warehouses: 6 rows
Created bronze_shipments: 36 rows
Created bronze_shipment_items: 43 rows
Created bronze_sensor_readings: 435 rows


In [4]:
from pyspark.sql.functions import (
    col, trim, to_timestamp, current_timestamp,
    input_file_name, when, lit, row_number
)
from pyspark.sql.window import Window

def overwrite_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print(f"Created {table_name}: {df.count()} rows")

# Products
silver_products = (
    spark.table("bronze_products")
    .select(
        trim(col("product_id")).alias("product_id"),
        trim(col("product_name")).alias("product_name"),
        trim(col("category")).alias("category"),
        col("min_temp_c").cast("double").alias("min_temp_c"),
        col("max_temp_c").cast("double").alias("max_temp_c"),
        col("unit_value_inr").cast("decimal(12,2)").alias("unit_value_inr"),
        col("shelf_life_hours").cast("int").alias("shelf_life_hours"),
        col("_ingested_at_utc")
    )
)

# Carriers
silver_carriers = (
    spark.table("bronze_carriers")
    .select(
        trim(col("carrier_id")).alias("carrier_id"),
        trim(col("carrier_name")).alias("carrier_name"),
        trim(col("service_level")).alias("service_level"),
        col("baseline_on_time_pct").cast("double").alias("baseline_on_time_pct"),
        col("baseline_temperature_compliance_pct").cast("double").alias(
            "baseline_temperature_compliance_pct"
        ),
        col("_ingested_at_utc")
    )
)

# Warehouses
silver_warehouses = (
    spark.table("bronze_warehouses")
    .select(
        trim(col("warehouse_id")).alias("warehouse_id"),
        trim(col("warehouse_name")).alias("warehouse_name"),
        trim(col("city")).alias("city"),
        trim(col("state")).alias("state"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        col("_ingested_at_utc")
    )
)

# Shipments
silver_shipments = (
    spark.table("bronze_shipments")
    .select(
        trim(col("shipment_id")).alias("shipment_id"),
        trim(col("carrier_id")).alias("carrier_id"),
        trim(col("origin_warehouse_id")).alias("origin_warehouse_id"),
        trim(col("destination_warehouse_id")).alias("destination_warehouse_id"),
        to_timestamp("planned_departure_utc").alias("planned_departure_utc"),
        to_timestamp("expected_arrival_utc").alias("expected_arrival_utc"),
        to_timestamp("actual_arrival_utc").alias("actual_arrival_utc"),
        trim(col("shipment_status")).alias("shipment_status"),
        trim(col("sensor_device_id")).alias("sensor_device_id"),
        trim(col("priority")).alias("priority"),
        col("_ingested_at_utc")
    )
)

# Shipment items
silver_shipment_items = (
    spark.table("bronze_shipment_items")
    .select(
        trim(col("shipment_item_id")).alias("shipment_item_id"),
        trim(col("shipment_id")).alias("shipment_id"),
        trim(col("product_id")).alias("product_id"),
        col("quantity").cast("int").alias("quantity"),
        col("declared_unit_value_inr").cast("decimal(12,2)").alias(
            "declared_unit_value_inr"
        ),
        col("_ingested_at_utc")
    )
)

overwrite_table(silver_products, "silver_products")
overwrite_table(silver_carriers, "silver_carriers")
overwrite_table(silver_warehouses, "silver_warehouses")
overwrite_table(silver_shipments, "silver_shipments")
overwrite_table(silver_shipment_items, "silver_shipment_items")

StatementMeta(, 8fb926b5-4034-4f27-b12c-299bf43bac80, 6, Finished, Available, Finished, False)

Created silver_products: 8 rows
Created silver_carriers: 4 rows
Created silver_warehouses: 6 rows
Created silver_shipments: 36 rows
Created silver_shipment_items: 43 rows


In [5]:
sensor_raw = (
    spark.table("bronze_sensor_readings")
    .select(
        trim(col("reading_id")).alias("reading_id"),
        trim(col("shipment_id")).alias("shipment_id"),
        trim(col("sensor_device_id")).alias("sensor_device_id"),
        to_timestamp("captured_at_utc").alias("captured_at_utc"),
        col("temperature_c").cast("double").alias("temperature_c"),
        col("humidity_pct").cast("double").alias("humidity_pct"),
        col("battery_pct").cast("double").alias("battery_pct"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        col("_ingested_at_utc")
    )
)

# Keep the first occurrence of a reading ID; mark later occurrences as duplicates.
duplicate_window = Window.partitionBy("reading_id").orderBy("_ingested_at_utc")

sensor_ranked = (
    sensor_raw
    .withColumn("duplicate_rank", row_number().over(duplicate_window))
)

valid_shipments = silver_shipments.select(
    col("shipment_id").alias("known_shipment_id")
)

sensor_checked = (
    sensor_ranked
    .join(
        valid_shipments,
        sensor_ranked.shipment_id == valid_shipments.known_shipment_id,
        "left"
    )
    .withColumn(
        "rejection_reason",
        when(
            col("shipment_id").isNull() | (trim(col("shipment_id")) == ""),
            lit("MISSING_SHIPMENT_ID")
        )
        .when(col("known_shipment_id").isNull(), lit("UNKNOWN_SHIPMENT_ID"))
        .when(col("temperature_c").isNull(), lit("MISSING_TEMPERATURE"))
        .when(
            (col("temperature_c") < -80) | (col("temperature_c") > 60),
            lit("IMPOSSIBLE_TEMPERATURE")
        )
        .when(col("duplicate_rank") > 1, lit("DUPLICATE_READING_ID"))
    )
)

rejected_sensor_readings = (
    sensor_checked
    .filter(col("rejection_reason").isNotNull())
    .withColumn("rejected_at_utc", current_timestamp())
    .drop("known_shipment_id")
)

silver_sensor_readings = (
    sensor_checked
    .filter(col("rejection_reason").isNull())
    .select(
        "reading_id",
        "shipment_id",
        "sensor_device_id",
        "captured_at_utc",
        "temperature_c",
        "humidity_pct",
        "battery_pct",
        "latitude",
        "longitude",
        "_ingested_at_utc"
    )
)

overwrite_table(rejected_sensor_readings, "rejected_sensor_readings")
overwrite_table(silver_sensor_readings, "silver_sensor_readings")

StatementMeta(, 8fb926b5-4034-4f27-b12c-299bf43bac80, 7, Finished, Available, Finished, False)

Created rejected_sensor_readings: 3 rows
Created silver_sensor_readings: 432 rows
